# 내시경 용종 분할의 실패 분석과 검수 우선순위

**Colab 런타임 → 런타임 유형 변경 → GPU → 모두 실행**

내시경 영상의 용종 분할 모델을 학습하고, 실패가 집중되는 형태를 검증 데이터에서 분석합니다. 이미지로부터 계산한 불확실성이 예측 실패와 인위적 주석 오류를 우선 검수하는 데 얼마나 도움이 되는지 측정합니다.

| 질문 | 평가 데이터 | 산출물 |
| --- | --- | --- |
| 어떤 크기·경계에서 분할이 실패하는가? | 검증 분석셋의 정답 | 형태별 성능·실패율, 사례 그림 |
| 같은 검수 예산에서 오류를 더 많이 발견하는가? | 검증 분석셋의 정답 및 합성 주입 기록 | 검수 큐, random 비교, AP·AUROC |
| 실패 분석에서 선택한 한 가지 손실 변경이 효과가 있는가? | 고정 테스트셋 | paired 성능 차이와 95% 구간 |

공식 배포 조건은 연구·교육 목적을 허용하며, 상업적 사용에는 사전 서면 허가가 필요합니다. 실험에 해당 데이터셋을 사용하거나 결과를 보고할 때 아래 논문을 인용합니다. [Simula 공식 배포 및 이용 조건](https://datasets.simula.no/kvasir-seg/)

Jha et al. (2020). *Kvasir-SEG: A Segmented Polyp Dataset*. MultiMedia Modeling, 451–462. [논문](https://doi.org/10.1007/978-3-030-37734-2_37)

상세 실험 설계와 재시작 방법은 함께 제공된 README에 있습니다. 이 프로젝트의 평가 범위는 공개 영상의 오프라인 분석입니다.


## 01 · 실행 설정

기본값은 Google Colab GPU와 공식 Kvasir-SEG를 사용합니다. 런타임 유형을 GPU로 변경하고 **모두 실행**을 누르세요. 브라우저에서 결과 ZIP 다운로드가 시작됩니다.

같은 런타임에서는 동일 설정으로 다시 실행하면 마지막 완료 epoch에서 이어집니다. 런타임이 삭제된 뒤에도 이어서 실행하려면 첫 실행부터 `MOUNT_DRIVE=True`로 설정하세요. Drive 인증은 Colab에서 진행합니다. `PROFILE="smoke"`는 CPU에서도 실행 가능한 합성 데이터 코드 점검입니다.

In [ ]:
from pathlib import Path
import importlib.metadata as metadata
import os
import subprocess
import sys

PROFILE = "research"  # "smoke" runs a small CPU/GPU integration check on generated fixtures.
WORK_ROOT_OVERRIDE = ""  # Optional, e.g. "/content/drive/MyDrive/polyp_audit"
LOCAL_ARCHIVE = ""  # Optional path to a copy of the official ZIP.
MOUNT_DRIVE = False
RESUME = True
DOWNLOAD_RESULTS = True
INCLUDE_RESUME_CHECKPOINTS_IN_ZIP = False
BATCH_SIZE = 8
NUM_WORKERS = 2

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
# Set before importing torch and before any CUDA context is created.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
ANALYSIS_PACKAGES = {
    "numpy": "2.3.5", "scipy": "1.17.0", "pandas": "2.2.3",
    "matplotlib": "3.10.8", "scikit-learn": "1.8.0", "Pillow": "13.0.0",
}
missing = []
for package, version in ANALYSIS_PACKAGES.items():
    try:
        metadata.version(package)
    except metadata.PackageNotFoundError:
        missing.append(f"{package}=={version}")
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)

import hashlib
import json
import math
import platform
import random
import shutil
import time
import urllib.request
import zipfile
from collections import defaultdict

import matplotlib.pyplot as plt
from IPython.display import display
import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage
from sklearn.metrics import average_precision_score, roc_auc_score

try:
    import torch
    import torchvision
    import torch.nn.functional as F
    from torch import nn
    from torch.utils.data import DataLoader, Dataset
    from torchvision.models.segmentation import (
        LRASPP_MobileNet_V3_Large_Weights, lraspp_mobilenet_v3_large,
    )
except (ImportError, RuntimeError) as error:
    raise RuntimeError(
        "호환되는 torch/torchvision이 필요합니다. Colab 기본 GPU 런타임으로 새로 연결하거나 "
        "로컬에서 공식 설치 안내의 일치하는 버전 조합을 설치하세요."
    ) from error

SMOKE = PROFILE == "smoke"
if PROFILE not in {"research", "smoke"}:
    raise ValueError("PROFILE은 research 또는 smoke로 설정하세요.")
if BATCH_SIZE < 2 or NUM_WORKERS < 0:
    raise ValueError("BATCH_SIZE는 2 이상, NUM_WORKERS는 0 이상이어야 합니다.")
if not SMOKE and not torch.cuda.is_available():
    raise RuntimeError("Colab의 런타임 유형을 GPU로 변경한 뒤 모두 실행하세요. CPU 검증은 PROFILE='smoke'입니다.")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
if MOUNT_DRIVE:
    if not IN_COLAB:
        raise ValueError("MOUNT_DRIVE는 Colab에서만 지원합니다.")
    from google.colab import drive
    drive.mount("/content/drive")

ROOT = Path(WORK_ROOT_OVERRIDE or (
    "/content/drive/MyDrive/polyp_audit" if MOUNT_DRIVE
    else "/content/polyp_audit" if IN_COLAB else "polyp_audit_work"
)).resolve()
# Keep training input and model download cache on local runtime disk.
CACHE = (Path("/content/polyp_audit_cache") if IN_COLAB else ROOT / "cache").resolve()
CACHE.mkdir(parents=True, exist_ok=True)
torch.hub.set_dir(str(CACHE / "torch_hub"))

CONFIG = {
    "protocol": "1.0", "profile": PROFILE, "seed": 2026,
    "side": 64 if SMOKE else 256, "epochs": 1 if SMOKE else 12,
    "batch_size": min(BATCH_SIZE, 4) if SMOKE else BATCH_SIZE,
    "workers": 0 if SMOKE else NUM_WORKERS,
    "amp": DEVICE.type == "cuda" and not SMOKE,
    "learning_rate": 5e-4, "backbone_learning_rate": 1e-4, "weight_decay": 1e-4,
    "probability_threshold": 0.50, "failure_dice": 0.60, "boundary_tolerance_px": 2,
    "upweight_factor": 4.0, "group_dhash_distance": 6, "group_rgb_mae": 0.08,
    "split_fractions": {"train": 0.70, "val_select": 0.10, "val_audit": 0.10, "test": 0.10},
    "review_budgets": [0.1, 0.2, 0.3, 0.5], "random_repeats": 500,
    "injection_repeats": 12, "injection_fraction": 0.50,
    "injection_types": ["shift", "erosion", "dilation", "missing"],
    "bootstrap_repeats": 2000, "tta_flips": [[], [-1], [-2], [-2, -1]],
    "entropy_top_fraction": 0.05, "pretrained": not SMOKE,
    "weights": "COCO_WITH_VOC_LABELS_V1" if not SMOKE else None,
    "mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225],
}
ENVIRONMENT = {
    "python": platform.python_version(), "platform": platform.platform(),
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name() if DEVICE.type == "cuda" else None,
    "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
    "versions": {p: metadata.version(p) for p in ["torch", "torchvision", *ANALYSIS_PACKAGES]},
}

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.use_deterministic_algorithms(True, warn_only=True)

def sha256_file(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def write_json(path, value):
    temporary = path.with_suffix(path.suffix + ".part")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
                         encoding="utf-8")
    temporary.replace(path)

def save_csv(frame, filename):
    target = RUN / filename
    temporary = target.with_suffix(".csv.part")
    frame.to_csv(temporary, index=False)
    temporary.replace(target)

def save_figure(fig, filename):
    fig.savefig(RUN / "figures" / filename, dpi=160, bbox_inches="tight", facecolor="white")
    plt.close(fig)

seed_everything(CONFIG["seed"])
print(f"Profile={PROFILE} | device={ENVIRONMENT['gpu'] or DEVICE} | torch={torch.__version__}")


## 02 · 공식 데이터 확보

Simula 배포처의 ZIP에서 `images/`와 `masks/`를 찾아 파일명으로 1,000쌍을 연결합니다. 배포본에는 bounding box JSON도 포함됩니다. 주 URL이 실패하면 Simula의 이전 배포 경로와 원저자의 Kaggle 배포본(v3)을 차례대로 시도합니다. 공식 원본을 미리 받은 경우 `LOCAL_ARCHIVE`를 사용할 수 있습니다.

In [ ]:
OFFICIAL_URLS = [
    "https://datasets.simula.no/downloads/kvasir-seg.zip",
    "https://datasets.simula.no/kvasir-seg/Kvasir-SEG.zip",
    "https://www.kaggle.com/api/v1/datasets/download/debeshjha1/kvasirseg?datasetVersionNumber=3",
]
OFFICIAL_PAGE = "https://datasets.simula.no/kvasir-seg/"
PAPER = "https://doi.org/10.1007/978-3-030-37734-2_37"
EXPECTED_ARCHIVE_SHA256 = "03b30e21d584e04facf49397a2576738fd626815771afbbf788f74a7153478f7"
# The checksum was recorded for the primary URL by torch-em. Record the hash of every run.
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp"}

def pair_files(root, expected):
    def index(folder):
        result = {}
        for path in sorted(folder.iterdir()):
            if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES:
                if path.stem in result:
                    raise ValueError(f"중복 샘플 ID: {path.stem}")
                result[path.stem] = path
        return result

    images, masks = (index(root / directory) for directory in ("images", "masks"))
    if set(images) != set(masks):
        raise ValueError(f"이미지/마스크 ID 불일치: {len(set(images) ^ set(masks))}개")
    if len(images) != expected:
        raise ValueError(f"예상 {expected}쌍, 실제 {len(images)}쌍")
    return [(key, images[key], masks[key]) for key in sorted(images)]

def zip_root(archive, destination):
    with zipfile.ZipFile(archive) as package:
        members = package.infolist()
        if len(members) > 5000 or sum(item.file_size for item in members) > 500_000_000:
            raise ValueError("Kvasir-SEG ZIP의 예상 크기를 초과합니다.")
        for member in members:
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination.resolve()):
                raise ValueError(f"안전하지 않은 ZIP 경로: {member.filename}")
        package.extractall(destination)
    roots = [folder for folder in destination.rglob("images")
             if folder.is_dir() and (folder.parent / "masks").is_dir()
             and "__MACOSX" not in folder.parts]
    if len(roots) != 1:
        raise ValueError("images/와 masks/가 있는 데이터 폴더를 하나 찾을 수 있어야 합니다.")
    return roots[0].parent

def get_official_data():
    default_archive = CACHE / "kvasir-seg.zip"
    uploaded = Path(LOCAL_ARCHIVE).expanduser() if LOCAL_ARCHIVE else None
    if uploaded is not None and not uploaded.is_file():
        raise FileNotFoundError(f"LOCAL_ARCHIVE의 공식 ZIP을 찾을 수 없습니다: {uploaded}")
    sources = [(uploaded, "user_supplied_official_archive")] if uploaded else []
    if not uploaded:
        if default_archive.exists():
            sources.append((default_archive, "cached_official_archive"))
        sources.extend((default_archive, url) for url in OFFICIAL_URLS)
    cached_provenance = CACHE / "kvasir-seg.source.json"
    errors = []
    for archive, source in sources:
        attempts = 2 if source.startswith("https://") else 1
        for attempt in range(attempts):
            partial = CACHE / "kvasir-seg.zip.part"
            staging = CACHE / "extracting"
            try:
                if source.startswith("https://"):
                    request = urllib.request.Request(source, headers={"User-Agent": "KvasirSEG-Research/1.0"})
                    with urllib.request.urlopen(request, timeout=35) as response, partial.open("wb") as stream:
                        shutil.copyfileobj(response, stream, 1024 * 1024)
                    file_to_verify = partial
                else:
                    file_to_verify = archive
                fingerprint = sha256_file(file_to_verify)
                if source == OFFICIAL_URLS[0] and fingerprint != EXPECTED_ARCHIVE_SHA256:
                    raise ValueError(f"공개 기록과 기본 ZIP의 SHA-256 불일치: {fingerprint}")
                if fingerprint != EXPECTED_ARCHIVE_SHA256:
                    print(f"기본 ZIP과 다른 SHA-256 ({source}): {fingerprint}. 구조 검사를 계속합니다.")
                extracted = CACHE / f"kvasir_{fingerprint}"
                if not extracted.exists():
                    shutil.rmtree(staging, ignore_errors=True)
                    staging.mkdir()
                    origin = zip_root(file_to_verify, staging)
                    pair_files(origin, expected=1000)
                    origin.replace(extracted)
                    shutil.rmtree(staging, ignore_errors=True)
                pairs = pair_files(extracted, expected=1000)
                provenance = {"source": source, "archive_sha256": fingerprint,
                              "matches_simula_primary_sha256": fingerprint == EXPECTED_ARCHIVE_SHA256}
                if file_to_verify == partial:
                    partial.replace(default_archive)
                    write_json(cached_provenance, provenance)
                elif source == "cached_official_archive" and cached_provenance.is_file():
                    recorded = json.loads(cached_provenance.read_text(encoding="utf-8"))
                    if recorded.get("archive_sha256") == fingerprint:
                        provenance["source"] = recorded["source"]
                return pairs, provenance
            except (OSError, ValueError, RuntimeError, zipfile.BadZipFile) as error:
                errors.append(f"{source} ({attempt + 1}/{attempts}): {error}")
                partial.unlink(missing_ok=True)
                shutil.rmtree(staging, ignore_errors=True)
                if attempt + 1 < attempts:
                    time.sleep(2)
    raise RuntimeError("공식 Kvasir-SEG ZIP 준비 실패. Simula 또는 저자의 Kaggle 배포본 ZIP을 내려받아 Colab에 "
                       "업로드한 뒤 LOCAL_ARCHIVE를 지정하고 다시 실행하세요.\n" + "\n".join(errors))

def smoke_fixtures():
    root = CACHE / "smoke_fixture_v1"
    for name in ("images", "masks"):
        (root / name).mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(CONFIG["seed"])
    y, x = np.mgrid[:72, :88]
    for index in range(48):
        height = 72 if index % 2 else 60
        cx, cy = rng.integers([23, 20], [64, 41])
        rx, ry = rng.integers([4, 4], [23, 18])
        mask = ((x - cx) / rx) ** 2 + ((y - cy) / ry) ** 2 < 1
        if index % 3 == 0:
            mask |= ((x - cx - 0.7 * rx) ** 2 + (y - cy) ** 2 < (ry * 0.65) ** 2)
        rgb = rng.integers(20, 180, (72, 88, 3), dtype=np.uint8)
        rgb[mask, 0] = 225
        key = f"fixture_{index:03d}.png"
        Image.fromarray(rgb[:height]).save(root / "images" / key)
        Image.fromarray(mask[:height].astype("uint8") * 255).save(root / "masks" / key)
    return pair_files(root, 48), {"source": "synthetic_smoke_fixtures", "archive_sha256": None}

PAIRS, DATA_INFO = smoke_fixtures() if SMOKE else get_official_data()
print(f"Paired files: {len(PAIRS)} | source: {DATA_INFO['source']}")


## 03 · 이미지 그룹 분할

정답 픽셀을 읽기 전에 정확·근접 중복 이미지의 연결 성분을 하나의 그룹으로 묶습니다. 분할 목표는 train 70%, val_select 10%, val_audit 10%, test 10%입니다. 그룹 보존을 위해 실제 수는 달라질 수 있습니다. 공개본에는 환자·검사 세션 ID가 없어 이미지 유사도로 확인 가능한 중복을 관리합니다.

In [ ]:
def image_fingerprint(path):
    with Image.open(path) as image:
        image = image.convert("RGB")
        digest = hashlib.sha256(f"{image.size}".encode() + image.tobytes()).hexdigest()
        thumb = np.asarray(image.resize((32, 32), Image.Resampling.BILINEAR), dtype=np.int16)
        gray = np.asarray(image.convert("L").resize((9, 9), Image.Resampling.BILINEAR))
    horizontal = gray[:8, 1:] > gray[:8, :-1]
    vertical = gray[1:, :8] > gray[:-1, :8]
    dhash = int.from_bytes(np.packbits(np.r_[horizontal.ravel(), vertical.ravel()]).tobytes(), "big")
    return digest, dhash, thumb

def grouped_split(pairs):
    fingerprints = [image_fingerprint(image) for _, image, _ in pairs]
    parents = list(range(len(pairs)))

    def root(index):
        while parents[index] != index:
            parents[index] = parents[parents[index]]
            index = parents[index]
        return index

    edges = []
    for i, (digest, dhash, thumb) in enumerate(fingerprints):
        for j in range(i):
            other_digest, other_hash, other_thumb = fingerprints[j]
            distance = (dhash ^ other_hash).bit_count()
            if digest != other_digest and distance > CONFIG["group_dhash_distance"]:
                continue
            mae = float(np.abs(thumb - other_thumb).mean() / 255)
            if digest == other_digest or mae <= CONFIG["group_rgb_mae"]:
                a, b = root(i), root(j)
                parents[max(a, b)] = min(a, b)
                edges.append((pairs[j][0], pairs[i][0], distance, mae, digest == other_digest))
    groups = defaultdict(list)
    for index, (key, _, _) in enumerate(pairs):
        groups[pairs[root(index)][0]].append(key)

    fractions = CONFIG["split_fractions"]
    names = list(fractions)
    targets = np.array(list(fractions.values())) * len(pairs)
    counts = np.zeros(len(names), dtype=int)
    rng = np.random.default_rng(CONFIG["seed"])
    group_ids = rng.permutation(sorted(groups)).tolist()
    group_ids.sort(key=lambda group: -len(groups[group]))  # Stable order after seeded permutation.
    rows = []
    for group in group_ids:
        slot = int(np.argmax(targets - counts))
        rows.extend((key, group, names[slot]) for key in groups[group])
        counts[slot] += len(groups[group])

    split = pd.DataFrame(rows, columns=["id", "group_id", "split"]).sort_values("id")
    split["decoded_rgb_sha256"] = [item[0] for item in fingerprints]
    split = split.reset_index(drop=True)
    assert split.id.is_unique and len(split) == len(pairs)
    assert set(split.split) == set(names) and split.groupby("group_id").split.nunique().max() == 1
    assert split.groupby("decoded_rgb_sha256").split.nunique().max() == 1
    edge_table = pd.DataFrame(edges, columns=["id_a", "id_b", "dhash_distance", "rgb_mae", "exact"])
    return split, edge_table

SPLIT, DUPLICATE_EDGES = grouped_split(PAIRS)
SPLIT_SHA = hashlib.sha256(SPLIT.to_csv(index=False).encode()).hexdigest()
RUN_SIGNATURE = {
    "config": CONFIG, "split_sha256": SPLIT_SHA, "archive_sha256": DATA_INFO["archive_sha256"],
    "versions": ENVIRONMENT["versions"], "device_type": DEVICE.type,
}
RUN_ID = hashlib.sha256(json.dumps(RUN_SIGNATURE, sort_keys=True).encode()).hexdigest()[:12]
RUN = ROOT / "runs" / f"{PROFILE}_{RUN_ID}"
for directory in (RUN, RUN / "figures", RUN / "checkpoints"):
    directory.mkdir(parents=True, exist_ok=True)
write_json(RUN / "run_setup.json", {
    "run_id": RUN_ID, **RUN_SIGNATURE, "environment": ENVIRONMENT, "dataset": DATA_INFO,
    "official_urls": OFFICIAL_URLS, "dataset_paper": PAPER, "dataset_homepage": OFFICIAL_PAGE,
    "policies": {
        "split_unit": "Connected components of exact or near-duplicate RGB images",
        "selection_metric": "Highest macro validation Dice, earliest epoch on tie",
        "primary_test_metric": "Paired delta in macro Dice (intervention minus baseline)",
        "mask_binarization": "L channel >= 128 after nearest-neighbor geometry preprocessing",
    },
})
save_csv(SPLIT, "split_manifest.csv")
save_csv(DUPLICATE_EDGES, "duplicate_candidates.csv")
split_summary = SPLIT.groupby("split", sort=False).agg(images=("id", "size"), groups=("group_id", "nunique"))
save_csv(split_summary.reset_index(), "split_summary.csv")
print(split_summary.to_string())
print(f"Run directory: {RUN}")


## 04 · 전처리와 학습 데이터 통계

종횡비를 유지한 letterbox를 적용합니다. 입력 영상은 bilinear, 마스크는 nearest-neighbor 보간 후 이진화합니다. 패딩은 손실과 지표에서 제외합니다. 크기와 경계 복잡도의 삼분위 기준은 학습 마스크에서만 계산합니다.

In [ ]:
def letterbox(image, mask, side):
    width, height = image.size
    ratio = min(side / width, side / height)
    w, h = max(1, round(width * ratio)), max(1, round(height * ratio))
    x, y = (side - w) // 2, (side - h) // 2
    rgb = np.full((side, side, 3), [124, 116, 104], dtype=np.uint8)
    target = np.zeros((side, side), dtype=bool)
    valid = np.zeros_like(target)
    roi = np.s_[y:y + h, x:x + w]
    rgb[roi] = np.asarray(image.resize((w, h), Image.Resampling.BILINEAR))
    target[roi] = np.asarray(mask.resize((w, h), Image.Resampling.NEAREST)) >= 128
    valid[roi] = True
    return rgb, target, valid

def load_pair(key, image_path, mask_path):
    try:
        with Image.open(image_path) as image, Image.open(mask_path) as mask:
            if image.size != mask.size:
                raise ValueError(f"이미지 크기 {image.size}, 마스크 크기 {mask.size}")
            return letterbox(image.convert("RGB"), mask.convert("L"), CONFIG["side"])
    except (OSError, ValueError) as error:
        raise RuntimeError(f"{key} 전처리 실패: {error}") from error

MEAN = np.asarray(CONFIG["mean"], dtype=np.float32)
STD = np.asarray(CONFIG["std"], dtype=np.float32)
PAIR_BY_ID = {key: (key, image, mask) for key, image, mask in PAIRS}
SPLIT_IDS = {name: SPLIT.loc[SPLIT.split.eq(name), "id"].tolist() for name in CONFIG["split_fractions"]}

class PolypDataset(Dataset):
    def __init__(self, keys, augment=False, epoch=0):
        self.pairs = [PAIR_BY_ID[key] for key in keys]
        self.augment = augment
        self.epoch = epoch

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        key, image_path, mask_path = self.pairs[index]
        rgb, target, valid = load_pair(key, image_path, mask_path)
        if self.augment:
            rng = np.random.default_rng([CONFIG["seed"], self.epoch, index])
            for axis in (0, 1):
                if rng.random() < 0.5:
                    rgb, target, valid = (np.flip(item, axis) for item in (rgb, target, valid))
            adjusted = np.clip(rgb.astype(np.float32) * rng.uniform(0.90, 1.10), 0, 255)
            rgb = np.where(valid[..., None], adjusted, rgb)
        normalized = (rgb.astype(np.float32) / 255.0 - MEAN) / STD
        return (torch.from_numpy(normalized.transpose(2, 0, 1).copy()),
                torch.from_numpy(target[None].copy()), torch.from_numpy(valid[None].copy()), key)

def loader(split, epoch=0):
    training = split == "train"
    generator = torch.Generator().manual_seed(CONFIG["seed"] + epoch)
    return DataLoader(PolypDataset(SPLIT_IDS[split], training, epoch), batch_size=CONFIG["batch_size"],
                      shuffle=training, num_workers=CONFIG["workers"], generator=generator,
                      pin_memory=DEVICE.type == "cuda")

def mask_geometry(target, valid):
    target = target & valid
    # Four-connected exposed grid edges give a perimeter proxy in input pixels.
    padded = np.pad(target, 1).astype(np.int16)
    perimeter = np.abs(np.diff(padded, axis=0)).sum() + np.abs(np.diff(padded, axis=1)).sum()
    area = int(target.sum())
    components = int(ndimage.label(target)[1])
    return {
        "area_fraction": area / int(valid.sum()), "area_pixels": area,
        "boundary_complexity": float(perimeter ** 2 / (4 * np.pi * area)) if area else np.nan,
        "components": components,
        "touches_valid_edge": bool((target & ~ndimage.binary_erosion(valid)).any()),
    }

train_geometry = pd.DataFrame([
    {"id": key, **mask_geometry(*load_pair(*PAIR_BY_ID[key])[1:])} for key in SPLIT_IDS["train"]
])
save_csv(train_geometry, "train_geometry.csv")
REFERENCE_CUTS = {
    metric: train_geometry.loc[train_geometry.area_pixels.gt(0), metric].quantile([1 / 3, 2 / 3]).tolist()
    for metric in ("area_fraction", "boundary_complexity")
}
if not all(np.isfinite(cuts).all() for cuts in REFERENCE_CUTS.values()):
    raise ValueError("학습용 마스크에서 유효한 크기·경계 통계를 계산할 수 없습니다.")
write_json(RUN / "geometry_cutoffs.json", REFERENCE_CUTS)


## 05 · 사전학습 모델과 평가 함수

COCO의 VOC 클래스 부분집합으로 학습된 TorchVision LR-ASPP MobileNetV3-Large를 불러와 두 최종 분류기를 1채널로 교체합니다. 기준 손실은 BCE 0.5 + soft Dice loss 0.5입니다. 정답과 단일 추론 예측을 비교해 이미지별 Dice, IoU, Recall, Precision을 계산합니다. 경계 F1의 허용 거리는 전처리 좌표 기준 2픽셀입니다.

In [ ]:
def build_model(pretrained=False):
    weights = LRASPP_MobileNet_V3_Large_Weights.COCO_WITH_VOC_LABELS_V1 if pretrained else None
    try:
        model = lraspp_mobilenet_v3_large(weights=weights, weights_backbone=None)
    except (OSError, RuntimeError) as error:
        raise RuntimeError("TorchVision 사전학습 가중치/API 초기화 실패. 연결 상태와 "
                           "torch/torchvision 버전 조합을 확인하고 다시 실행하세요.") from error
    for name in ("low_classifier", "high_classifier"):
        old = getattr(model.classifier, name)
        setattr(model.classifier, name, nn.Conv2d(old.in_channels, 1, kernel_size=1))
    return model

def save_checkpoint(path, payload):
    temp = path.with_suffix(".pt.part")
    torch.save(payload, temp)
    temp.replace(path)

def initialization():
    path = RUN / "checkpoints" / "shared_initialization.pt"
    if not path.exists() or not RESUME:
        seed_everything(CONFIG["seed"])
        model = build_model(CONFIG["pretrained"]).eval()
        with torch.inference_mode():
            side = CONFIG["side"]
            output = model.to(DEVICE)(torch.zeros((1, 3, side, side), device=DEVICE))["out"]
        assert output.shape == (1, 1, side, side) and torch.isfinite(output).all()
        save_checkpoint(path, model.cpu().state_dict())
    return path

INITIAL_PATH = initialization()
INIT_SHA = sha256_file(INITIAL_PATH)
N_PARAMETERS = sum(tensor.numel() for name, tensor in torch.load(
    INITIAL_PATH, weights_only=True, map_location="cpu").items()
    if name.endswith((".weight", ".bias")))
print(f"LR-ASPP binary model: {N_PARAMETERS:,} parameters.")

def metric_rows(predictions, targets, valid_pixels, keys, detailed=False):
    def ratio(numerator, denominator, empty):
        return numerator / denominator if denominator else float(empty)

    def boundary_f1(prediction, target, valid):
        a = prediction & ~ndimage.binary_erosion(prediction)
        b = target & ~ndimage.binary_erosion(target)
        if not a.any() or not b.any():
            return float(a.any() == b.any())
        tolerance = CONFIG["boundary_tolerance_px"]
        match_a = (a & valid & (ndimage.distance_transform_edt(~b) <= tolerance)).sum() / a.sum()
        match_b = (b & valid & (ndimage.distance_transform_edt(~a) <= tolerance)).sum() / b.sum()
        return ratio(2 * match_a * match_b, match_a + match_b, False)

    rows = []
    for key, pred, target, valid in zip(keys, predictions, targets, valid_pixels, strict=True):
        pred, target = pred & valid, target & valid
        tp = int((pred & target).sum())
        fp = int((pred & ~target).sum())
        fn = int((~pred & target).sum())
        tn = int(valid.sum() - tp - fp - fn)
        row = {
            "id": key, "tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "dice": ratio(2 * tp, 2 * tp + fp + fn, True),
            "iou": ratio(tp, tp + fp + fn, True),
            "recall": ratio(tp, tp + fn, True),
            "precision": ratio(tp, tp + fp, not target.any()),
        }
        if detailed:
            row.update(mask_geometry(target, valid))
            row["boundary_f1"] = boundary_f1(pred, target, valid)
            row["failure"] = row["dice"] < CONFIG["failure_dice"]
            row["missed_entire_polyp"] = bool(target.any() and not pred.any())
        rows.append(row)
    return rows

@torch.inference_mode()
def evaluate(model, split, detailed=False):
    model.eval()
    rows = []
    for images, targets, valid, keys in loader(split):
        probs = model(images.to(DEVICE, non_blocking=True))["out"].float().sigmoid().cpu()
        rows.extend(metric_rows(probs[:, 0].numpy() >= CONFIG["probability_threshold"],
                                targets[:, 0].numpy(), valid[:, 0].numpy(), keys, detailed))
    return pd.DataFrame(rows)

def objective(logits, target, valid, mode):
    target, valid = target.float(), valid.float()
    logits = logits.float()
    probability = logits.sigmoid() * valid
    dimensions = (1, 2, 3)
    area = target.sum(dim=dimensions)
    intersection = (probability * target).sum(dim=dimensions)
    dice_loss = 1.0 - ((2 * intersection + 1) / (probability.sum(dim=dimensions) + area + 1))
    weight = valid
    if mode == "small_positive":
        small = area / valid.sum(dim=dimensions) <= REFERENCE_CUTS["area_fraction"][0]
        emphasis = target * small[:, None, None, None]
        weight = weight * (1 + (CONFIG["upweight_factor"] - 1) * emphasis)
    elif mode == "boundary":
        dilated = F.max_pool2d(target, 5, stride=1, padding=2)
        eroded = 1 - F.max_pool2d(F.pad(1 - target, (2, 2, 2, 2), value=1), 5, stride=1)
        weight = weight * (1 + (CONFIG["upweight_factor"] - 1) * (dilated - eroded))
    pixel_bce = F.binary_cross_entropy_with_logits(logits, target, reduction="none")
    bce = (pixel_bce * weight).sum(dim=dimensions) / weight.sum(dim=dimensions)
    return (0.5 * bce + 0.5 * dice_loss).mean()


## 06 · 기준 모델 학습 및 복구

두 모델은 동일한 초기 가중치와 seed별 학습 순서를 사용합니다. 매 epoch 후 체크포인트를 저장합니다. val_select의 macro Dice가 가장 높은 가중치를 선택하며, 동점이면 이른 epoch를 유지합니다.

In [ ]:
def load_best(name):
    path = RUN / "checkpoints" / f"{name}_best.pt"
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    if checkpoint["run_id"] != RUN_ID or checkpoint["initial_sha256"] != INIT_SHA:
        raise RuntimeError(f"체크포인트와 현재 실행 설정이 일치하지 않습니다: {path}")
    model = build_model()
    model.load_state_dict(checkpoint["model"])
    return model.to(DEVICE).eval(), checkpoint["epoch"]

def fit(name, mode):
    latest = RUN / "checkpoints" / f"{name}_last.pt"
    best = RUN / "checkpoints" / f"{name}_best.pt"
    model = build_model()
    model.load_state_dict(torch.load(INITIAL_PATH, map_location="cpu", weights_only=True))
    model.to(DEVICE)
    optimizer = torch.optim.AdamW([
        {"params": model.backbone.parameters(), "lr": CONFIG["backbone_learning_rate"]},
        {"params": model.classifier.parameters(), "lr": CONFIG["learning_rate"]},
    ], weight_decay=CONFIG["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=CONFIG["amp"])
    history, first_epoch = [], 1
    best_dice, best_epoch, best_model = -1.0, None, None
    identity = {"run_id": RUN_ID, "initial_sha256": INIT_SHA, "mode": mode}
    if RESUME and latest.exists():
        state = torch.load(latest, weights_only=True, map_location="cpu")
        if any(state[key] != value for key, value in identity.items()):
            raise RuntimeError(f"재개 설정 불일치: {latest}. 새 작업 경로를 사용하거나 RESUME=False로 재실행하세요.")
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scaler.load_state_dict(state["scaler"])
        history, first_epoch = state["history"], state["epoch"] + 1
        best_dice, best_epoch, best_model = state["best_dice"], state["best_epoch"], state["best_model"]
        del state
        print(f"{name}: completed {first_epoch - 1} epoch(s), resuming.")
    try:
        for epoch in range(first_epoch, CONFIG["epochs"] + 1):
            seed_everything(CONFIG["seed"] + epoch)
            model.train()
            total_loss = 0.0
            for images, targets, valid, _ in loader("train", epoch):
                images, targets, valid = [value.to(DEVICE, non_blocking=True)
                                          for value in (images, targets, valid)]
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=CONFIG["amp"]):
                    logits = model(images)["out"]
                loss = objective(logits, targets, valid, mode)
                if not torch.isfinite(loss):
                    raise FloatingPointError(f"{name}, epoch {epoch}: non-finite loss")
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                total_loss += float(loss.detach()) * len(images)

            selection = evaluate(model, "val_select")
            means = selection[["dice", "iou", "recall", "precision"]].mean()
            history.append({"epoch": epoch, "train_loss": total_loss / len(SPLIT_IDS["train"]),
                            **{f"val_{metric}": float(value) for metric, value in means.items()}})
            if means.dice > best_dice:
                best_dice, best_epoch = float(means.dice), epoch
                best_model = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            save_checkpoint(latest, {**identity, "epoch": epoch, "model": model.state_dict(),
                                     "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),
                                     "best_model": best_model, "best_dice": best_dice,
                                     "best_epoch": best_epoch, "history": history})
            save_csv(pd.DataFrame(history), f"{name}_history.csv")
            print(f"{name:12} epoch {epoch:02}/{CONFIG['epochs']}  "
                  f"loss={history[-1]['train_loss']:.4f}  val Dice={means.dice:.4f}", flush=True)
    except torch.cuda.OutOfMemoryError as error:
        raise RuntimeError("GPU 메모리 부족. 런타임을 다시 시작하고 BATCH_SIZE=4 또는 2로 새 실험을 실행하세요. "
                           f"동일 설정으로 재개할 때 사용할 완료 epoch 체크포인트: {latest}") from error
    except (RuntimeError, OSError, FloatingPointError, KeyboardInterrupt) as error:
        print(f"훈련 중단. 같은 설정으로 다시 실행하면 마지막 완료 epoch부터 재개합니다: {latest}", flush=True)
        raise
    save_checkpoint(best, {**identity, "epoch": best_epoch, "val_dice": best_dice, "model": best_model})
    history = pd.DataFrame(history)
    save_csv(history, f"{name}_history.csv")
    return history

BASELINE_HISTORY = fit("baseline", "baseline")
BASELINE, BASELINE_EPOCH = load_best("baseline")


## 07 · 검증 실패 분석과 가설 선택

val_audit에서 Dice < 0.60을 실패로 정의합니다. 크기, 경계 복잡도, 이미지 경계 접촉, 마스크 연결 성분 수별로 실패 양상을 집계합니다. 작은 용종군과 복잡한 경계군의 기준 모델 Dice를 각 나머지 군과 비교합니다. 사전에 정의한 규칙이 둘 중 한 가설을 선택합니다.

In [ ]:
GEOMETRY_STAGES = {
    "area_fraction": ["small", "medium", "large"],
    "boundary_complexity": ["simple", "intermediate", "complex"],
}

def assign_geometry(frame):
    result = frame.copy()
    for feature, labels in GEOMETRY_STAGES.items():
        low, high = REFERENCE_CUTS[feature]
        group = np.select([result[feature].le(low), result[feature].le(high)], labels[:2], default=labels[2])
        result[f"{feature}_group"] = np.where(result.area_pixels.gt(0), group, "empty_reference")
    return result

def group_diagnostics(frame):
    tables = []
    for feature in [*GEOMETRY_STAGES, "touches_valid_edge", "components"]:
        column = f"{feature}_group" if feature in GEOMETRY_STAGES else feature
        grouped = frame.groupby(column, sort=True)
        table = grouped.agg(
            n=("id", "size"), mean_dice=("dice", "mean"), failure_rate=("failure", "mean"),
            mean_iou=("iou", "mean"), mean_recall=("recall", "mean"),
            mean_precision=("precision", "mean"), mean_boundary_f1=("boundary_f1", "mean"),
            complete_miss_rate=("missed_entire_polyp", "mean"), fp_pixels=("fp", "sum"), fn_pixels=("fn", "sum"),
        ).reset_index(names="group")
        table.insert(0, "feature", feature)
        tables.append(table)
    return pd.concat(tables, ignore_index=True)

BASELINE_AUDIT = assign_geometry(evaluate(BASELINE, "val_audit", detailed=True))
save_csv(BASELINE_AUDIT, "baseline_val_audit_per_image.csv")
DIAGNOSTICS = group_diagnostics(BASELINE_AUDIT)
save_csv(DIAGNOSTICS, "validation_failure_by_geometry.csv")

fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
for ax, (feature, labels) in zip(axes, GEOMETRY_STAGES.items()):
    subset = DIAGNOSTICS[DIAGNOSTICS.feature.eq(feature)].set_index("group").reindex(labels)
    positions = np.arange(len(labels))
    for offset, metric, name, color in (
        (-0.18, "mean_dice", "Mean Dice", "#087E8B"),
        (0.18, "failure_rate", "Failure rate", "#CF5C36"),
    ):
        ax.bar(positions + offset, subset[metric], width=0.34, label=name, color=color)
    ax.set(ylim=(0, 1), title=feature.replace("_", " ").title(),
           xticks=positions, xticklabels=[f"{label}\nn={int(n)}" if pd.notna(n) else f"{label}\nn=0"
                                        for label, n in zip(labels, subset.n)])
    ax.legend(frameon=False)
save_figure(fig, "validation_failure_by_geometry.png")

def validation_gaps(frame):
    # Positive values mean the candidate subgroup had lower Dice than the rest.
    valid = frame.loc[frame.area_pixels.gt(0)]
    hypotheses = [("small_positive", "area_fraction_group", "small"),
                  ("boundary", "boundary_complexity_group", "complex")]
    evidence = {}
    for name, column, label in hypotheses:
        is_group = valid[column].eq(label)
        selected, others = valid.loc[is_group, "dice"], valid.loc[~is_group, "dice"]
        evidence[name] = {
            "group": label, "group_n": len(selected), "other_n": len(others),
            "group_dice": float(selected.mean()) if len(selected) else None,
            "other_dice": float(others.mean()) if len(others) else None,
            "dice_gap": float(others.mean() - selected.mean()) if len(selected) and len(others) else None,
        }
    return evidence

EVIDENCE = validation_gaps(BASELINE_AUDIT)
minimum_n = 1 if SMOKE else 8
eligible = {name: score["dice_gap"] for name, score in EVIDENCE.items()
            if min(score["group_n"], score["other_n"]) >= minimum_n}
MODE = max(eligible, key=eligible.get) if eligible else "boundary"
HYPOTHESIS = {
    "small_positive": "작은 용종의 전경 픽셀에 BCE 가중치를 높이면 작은 용종의 Recall이 상승할 것이다.",
    "boundary": "정답 윤곽 주변 픽셀에 BCE 가중치를 높이면 복잡한 경계의 Boundary F1이 상승할 것이다.",
}[MODE]
DECISION = {
    "mode": MODE, "hypothesis": HYPOTHESIS, "baseline_val_audit_evidence": EVIDENCE,
    "rule": "Largest signed subgroup Dice gap across eligible groups. Small-positive on tie; boundary fallback.",
    "minimum_subgroup_and_complement_n": minimum_n,
    "positive_gap_observed": bool(eligible and eligible[MODE] > 0),
}
write_json(RUN / "improvement_hypothesis.json", DECISION)
print(HYPOTHESIS, "\n", json.dumps(EVIDENCE, ensure_ascii=False, indent=2))


## 08 · 정답 없는 예측 검수 점수

좌우·상하 반전을 조합한 4회 추론을 원좌표로 되돌려 확률을 평균합니다. 유효 픽셀 중 상위 5%의 평균 예측 엔트로피로 순위를 생성합니다. 입력 영상만으로 점수화한 뒤, 오프라인 평가에서 Dice 실패 레이블을 조인합니다. 동일한 검수 예산에서 500개 무작위 순열과 비교합니다.

In [ ]:
class ReviewImages(Dataset):
    """Inference data source with image paths and no reference mask paths."""
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        key, path = self.items[index]
        with Image.open(path) as image:
            image = image.convert("RGB")
            rgb, _, valid = letterbox(image, Image.new("L", image.size), CONFIG["side"])
        normalized = (rgb.astype(np.float32) / 255.0 - MEAN) / STD
        return torch.from_numpy(normalized.transpose(2, 0, 1).copy()), torch.from_numpy(valid), key

@torch.inference_mode()
def infer_review_maps(model, image_items):
    model.eval()
    review_loader = DataLoader(ReviewImages(image_items), batch_size=CONFIG["batch_size"], shuffle=False)
    maps = {}
    for images, valid, keys in review_loader:
        images = images.to(DEVICE)
        tta_probs = []
        for axes in CONFIG["tta_flips"]:
            batch = images.flip(axes) if axes else images
            prob = model(batch)["out"].float().sigmoid()
            tta_probs.append(prob.flip(axes) if axes else prob)
        prob = torch.stack(tta_probs).mean(dim=0)
        safe = prob.clamp(1e-6, 1 - 1e-6)
        entropy = -(safe * safe.log2() + (1 - safe) * (1 - safe).log2())
        for key, probability, uncertainty, valid_mask in zip(
            keys, prob[:, 0].cpu().numpy(), entropy[:, 0].cpu().numpy(), valid.numpy(), strict=True
        ):
            maps[key] = {"p": probability, "entropy": uncertainty, "valid": valid_mask}
    return maps

def rank_predictions(maps):
    rows = []
    for key, values in maps.items():
        scores = values["entropy"][values["valid"]]
        k = max(1, math.ceil(CONFIG["entropy_top_fraction"] * scores.size))
        uncertainty = float(np.partition(scores, -k)[-k:].mean())
        rows.append({"id": key, "uncertainty_score": uncertainty})
    ranking = pd.DataFrame(rows).sort_values(["uncertainty_score", "id"], ascending=[False, True])
    return ranking.assign(review_rank=np.arange(1, len(ranking) + 1))

AUDIT_IMAGES = [(key, PAIR_BY_ID[key][1]) for key in SPLIT_IDS["val_audit"]]
REVIEW_MAPS = infer_review_maps(BASELINE, AUDIT_IMAGES)
RANKING = rank_predictions(REVIEW_MAPS)
save_csv(RANKING, "validation_prediction_review_queue.csv")

def ranking_labels(frame, score, flag):
    ordered = frame.sort_values([score, "id"], ascending=[False, True])
    return ordered[flag].to_numpy(dtype=bool)

def review_curves(labels, seed):
    labels = np.asarray(labels, dtype=bool)
    rng = np.random.default_rng(seed)
    orders = np.array([rng.permutation(labels) for _ in range(CONFIG["random_repeats"])])
    random_hits = orders.cumsum(axis=1)
    positives = int(labels.sum())
    k = np.arange(1, len(labels) + 1)
    hits = labels.cumsum()
    rand_average = random_hits.mean(axis=0)
    low, high = np.quantile(random_hits, [0.025, 0.975], axis=0)
    return pd.DataFrame({
        "reviewed": k, "review_fraction": k / len(labels), "errors": positives,
        "ranked_found": hits, "random_mean_found": rand_average,
        "ranked_recall": hits / positives if positives else np.nan,
        "random_mean_recall": rand_average / positives if positives else np.nan,
        "random_p025_recall": low / positives if positives else np.nan,
        "random_p975_recall": high / positives if positives else np.nan,
        "ranked_precision": hits / k, "random_mean_precision": rand_average / k,
        "lift_vs_random_mean": np.divide(hits, rand_average, out=np.full(len(k), np.nan), where=rand_average > 0),
    })

def budget_rows(curve):
    indices = [math.ceil(len(curve) * budget) - 1 for budget in CONFIG["review_budgets"]]
    return curve.iloc[indices].assign(planned_budget=CONFIG["review_budgets"]).reset_index(drop=True)

def discrimination(frame, score, flag):
    truth = frame[flag].astype(int)
    if truth.nunique() < 2:
        return {"average_precision": np.nan, "auroc": np.nan}
    return {"average_precision": average_precision_score(truth, frame[score]),
            "auroc": roc_auc_score(truth, frame[score])}

def plot_review(curves, filename, title, comparison=None):
    fig, ax = plt.subplots(figsize=(6.8, 4.5), layout="constrained")
    x = curves.review_fraction.to_numpy()
    ax.plot(np.r_[0, x], np.r_[0, curves.ranked_recall], color="#087E8B", lw=2, label="Review score")
    ax.plot(np.r_[0, x], np.r_[0, curves.random_mean_recall], color="#535B65",
            lw=1.5, ls="--", label="Random mean")
    if "random_p025_recall" in curves:
        ax.fill_between(x, curves.random_p025_recall.to_numpy(), curves.random_p975_recall.to_numpy(),
                        color="#535B65", alpha=0.15, label="Random 95% range")
    if comparison is not None:
        ax.plot(np.r_[0, x], np.r_[0, comparison], color="#CF5C36", lw=1.6, label="Disagreement only")
    if not curves.ranked_recall.notna().any():
        ax.text(0.5, 0.5, "No positive cases", ha="center", transform=ax.transAxes)
    ax.set(xlim=(0, 1), ylim=(0, 1.03), xlabel="Fraction reviewed", ylabel="Fraction of errors found", title=title)
    ax.legend(frameon=False, loc="lower right")
    save_figure(fig, filename)

# Offline evaluation starts after the queue has been fixed and saved.
NATURAL = RANKING.merge(BASELINE_AUDIT[["id", "dice", "failure"]], on="id", validate="one_to_one")
save_csv(NATURAL, "validation_prediction_review_evaluation.csv")
NATURAL_CURVE = review_curves(ranking_labels(NATURAL, "uncertainty_score", "failure"), CONFIG["seed"] + 100)
NATURAL_BUDGET = budget_rows(NATURAL_CURVE)
NATURAL_STATS = pd.DataFrame([{
    "n": len(NATURAL), "failures": int(NATURAL.failure.sum()), "prevalence": NATURAL.failure.mean(),
    **discrimination(NATURAL, "uncertainty_score", "failure"),
}])
save_csv(NATURAL_CURVE, "validation_prediction_review_curve.csv")
save_csv(NATURAL_BUDGET, "validation_prediction_review_budgets.csv")
save_csv(NATURAL_STATS, "validation_prediction_review_discrimination.csv")
plot_review(NATURAL_CURVE, "validation_prediction_review.png", "Prediction failures on validation audit")

def failure_montage(frame):
    worst = frame.nsmallest(min(6, len(frame)), "dice")
    fig, axes = plt.subplots(len(worst), 4, figsize=(9, 2.15 * len(worst)), squeeze=False, layout="constrained")
    for axes_row, row in zip(axes, worst.itertuples(), strict=True):
        rgb, target, valid = load_pair(*PAIR_BY_ID[row.id])
        # Evaluate and show the same single-pass predictions used for the failure labels.
        tensor = PolypDataset([row.id])[0][0][None].to(DEVICE)
        with torch.inference_mode():
            prediction = BASELINE(tensor)["out"][0, 0].sigmoid().cpu().numpy() >= CONFIG["probability_threshold"]
        error = np.zeros((*target.shape, 3), dtype=np.uint8)
        error[valid & prediction & ~target] = (236, 76, 53)
        error[valid & ~prediction & target] = (57, 150, 241)
        error[valid & prediction & target] = (103, 173, 125)
        images = [rgb, target, error, np.where(valid, REVIEW_MAPS[row.id]["entropy"], np.nan)]
        for ax, image, title in zip(
            axes_row, images, ["Input", f"Reference | Dice {row.dice:.3f}", "TP green / FP red / FN blue", "TTA entropy"],
            strict=True,
        ):
            if title == "TTA entropy":
                ax.imshow(image, cmap="magma", vmin=0, vmax=1)
            else:
                ax.imshow(image, cmap="gray" if title.startswith("Reference") else None, vmin=0, vmax=1)
            ax.set_title(title, fontsize=9)
            ax.axis("off")
    save_figure(fig, "validation_failure_examples.png")

failure_montage(BASELINE_AUDIT)
display(NATURAL_BUDGET[["planned_budget", "reviewed", "ranked_found", "random_mean_found", "ranked_recall"]])


## 09 · 합성 주석 오류 검출

val_audit에서 일부 후보 마스크에 이동, 침식, 팽창, 완전 삭제를 주입합니다. 점수는 후보 마스크와 영상에서 얻은 모델 출력을 받습니다. 오프라인에서만 주입 이력과 원본 참조를 사용해 정오를 결정합니다. 엔트로피·확신도 가중 불일치를 가중치 없는 불일치 및 무작위 검수와 비교합니다.

In [ ]:
def alter_mask(reference, valid, corruption, rng):
    area = int(reference.sum())
    radius = int(np.clip(round(np.sqrt(area / np.pi) * rng.uniform(0.12, 0.26)), 1, 20))
    if corruption == "shift":
        offset = int(np.clip(round(2.5 * radius), 2, 45))
        axis = int(rng.integers(0, 2))
        delta = [0, 0]
        delta[axis] = offset * int(rng.choice([-1, 1]))
        candidate = ndimage.shift(reference, delta, order=0, mode="constant", cval=0, prefilter=False)
    elif corruption == "erosion":
        candidate = ndimage.binary_erosion(reference, iterations=radius)
    elif corruption == "dilation":
        candidate = ndimage.binary_dilation(reference, iterations=radius)
    elif corruption == "missing":
        candidate = np.zeros_like(reference)
    elif corruption == "clean":
        candidate = reference.copy()
    else:
        raise ValueError(f"Unknown injection: {corruption}")
    return candidate & valid

def score_candidate(candidate, maps):
    """Review score uses only the candidate annotation and image-derived model maps."""
    candidate = candidate & maps["valid"]
    probability, entropy, valid = maps["p"], maps["entropy"], maps["valid"]
    predicted = (probability >= CONFIG["probability_threshold"]) & valid
    support = (candidate | predicted) & valid
    if not support.any():
        return {"disagreement_score": 0.0, "annotation_review_score": 0.0}
    disagreement = candidate ^ predicted
    confidence = 2 * np.abs(probability - 0.5)
    # Entropy enters only on disagreeing pixels. The denominator excludes easy distant background.
    weighted = disagreement * (0.75 * confidence + 0.25 * entropy)
    return {"disagreement_score": float(disagreement.sum() / support.sum()),
            "annotation_review_score": float(weighted[support].mean())}

references = {key: load_pair(*PAIR_BY_ID[key])[1] for key in SPLIT_IDS["val_audit"]}
synthetic_rows, synthetic_scores = [], []
for trial in range(1, CONFIG["injection_repeats"] + 1):
    rng = np.random.default_rng([CONFIG["seed"], 400, trial])
    keys = rng.permutation(sorted(references)).tolist()
    count = int(round(len(keys) * CONFIG["injection_fraction"]))
    corruption_order = rng.permutation(CONFIG["injection_types"]).tolist()
    for position, key in enumerate(keys):
        intended = corruption_order[position % len(corruption_order)] if position < count else "clean"
        reference, maps = references[key], REVIEW_MAPS[key]
        candidate = alter_mask(reference, maps["valid"], intended, rng)
        synthetic_scores.append({"trial": trial, "id": key, **score_candidate(candidate, maps)})
        # Ground-truth fields are assembled separately from the review score.
        changed_pixels = int((candidate ^ reference).sum())
        union = int((candidate | reference).sum())
        synthetic_rows.append({
            "trial": trial, "id": key, "requested_corruption": intended,
            "is_corrupted": bool(changed_pixels), "changed_pixels": changed_pixels,
            "reference_candidate_iou": float((candidate & reference).sum() / union) if union else 1.0,
        })

SYNTHETIC_QUEUE = pd.DataFrame(synthetic_scores).sort_values(
    ["trial", "annotation_review_score", "id"], ascending=[True, False, True]
)
SYNTHETIC_QUEUE["review_rank"] = SYNTHETIC_QUEUE.groupby("trial").cumcount() + 1
save_csv(SYNTHETIC_QUEUE, "validation_synthetic_annotation_review_queue.csv")
SYNTHETIC = SYNTHETIC_QUEUE.merge(pd.DataFrame(synthetic_rows), on=["trial", "id"], validate="one_to_one")
save_csv(SYNTHETIC, "validation_synthetic_annotation_per_case.csv")

trial_curves, trial_budgets, synthetic_stats, type_rows = [], [], [], []
for trial, cases in SYNTHETIC.groupby("trial"):
    for score in ("annotation_review_score", "disagreement_score"):
        curve = review_curves(ranking_labels(cases, score, "is_corrupted"), CONFIG["seed"] + trial + 900)
        trial_curves.append(curve.assign(trial=trial, scoring=score))
        trial_budgets.append(budget_rows(curve).assign(trial=trial, scoring=score))
        synthetic_stats.append({"trial": trial, "scoring": score, "n": len(cases),
                                "positives": int(cases.is_corrupted.sum()), "prevalence": cases.is_corrupted.mean(),
                                **discrimination(cases, score, "is_corrupted")})
    ordered = cases.sort_values("review_rank")
    top = ordered.head(math.ceil(0.2 * len(ordered)))
    for kind in CONFIG["injection_types"]:
        eligible_cases = ordered.requested_corruption.eq(kind) & ordered.is_corrupted
        discovered = int((top.requested_corruption.eq(kind) & top.is_corrupted).sum())
        n = int(eligible_cases.sum())
        type_rows.append({"trial": trial, "corruption": kind, "n": n, "top20_found": discovered,
                          "random_expected_found": n * len(top) / len(ordered),
                          "mean_candidate_iou": ordered.loc[eligible_cases, "reference_candidate_iou"].mean()})

SYNTHETIC_CURVES = pd.concat(trial_curves, ignore_index=True)
SYNTHETIC_BUDGETS = pd.concat(trial_budgets, ignore_index=True)
SYNTHETIC_STATS = pd.DataFrame(synthetic_stats)
SYNTHETIC_TYPES = pd.DataFrame(type_rows)
SYNTHETIC_MEANS = SYNTHETIC_BUDGETS.groupby(["scoring", "planned_budget"], as_index=False)[
    ["reviewed", "errors", "ranked_found", "random_mean_found", "ranked_recall", "random_mean_recall",
     "ranked_precision", "random_mean_precision", "lift_vs_random_mean"]
].mean()
for name, table in {
    "curve_trials": SYNTHETIC_CURVES,
    "budgets_trials": SYNTHETIC_BUDGETS,
    "budgets_mean": SYNTHETIC_MEANS,
    "discrimination": SYNTHETIC_STATS,
    "types_top20": SYNTHETIC_TYPES,
}.items():
    save_csv(table, f"validation_synthetic_annotation_{name}.csv")

# The repetitions reuse validation images. Report descriptive averages, without independent-sample CIs.
mean_curves = SYNTHETIC_CURVES.groupby(["scoring", "review_fraction"], as_index=False)[
    ["ranked_recall", "random_mean_recall"]
].mean()
plot_review(mean_curves[mean_curves.scoring.eq("annotation_review_score")],
            "validation_synthetic_annotation_review.png", "Synthetic annotation errors across trials",
            mean_curves.loc[mean_curves.scoring.eq("disagreement_score"), "ranked_recall"].to_numpy())
display(SYNTHETIC_STATS.groupby("scoring")[["prevalence", "average_precision", "auroc"]].mean())


## 10 · 통제된 개선 실험

07절에서 선정된 한 종류의 픽셀만 BCE 가중치를 4배로 설정합니다. 이미지별 가중 BCE 평균을 사용하며, 다른 학습 조건과 체크포인트 선택 규칙은 기준 모델과 같습니다.

In [ ]:
INTERVENTION_HISTORY = fit("intervention", MODE)
INTERVENTION, INTERVENTION_EPOCH = load_best("intervention")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.7), layout="constrained")
for label, history, color in (
    ("Baseline", BASELINE_HISTORY, "#2D4865"), ("Intervention", INTERVENTION_HISTORY, "#087E8B"),
):
    axes[0].plot(history.epoch, history.train_loss, "o-", ms=3, color=color, label=label)
    axes[1].plot(history.epoch, history.val_dice, "o-", ms=3, color=color, label=label)
axes[0].set(title="Training objective (different BCE weighting)", xlabel="Epoch", ylabel="Loss")
axes[1].set(title="Checkpoint selection on validation", xlabel="Epoch", ylabel="Macro Dice", ylim=(0, 1))
for ax in axes:
    ax.legend(frameon=False)
save_figure(fig, "training_curves.png")


## 11 · 잠금된 테스트 분할의 최종 비교

두 체크포인트와 평가 설정을 기록한 뒤, 두 모델을 고정 테스트 이미지의 같은 조건에서 평가합니다. 주 지표는 전체 Dice의 paired 차이입니다. 이미지 그룹을 재표집하는 bootstrap으로 95% 구간을 구합니다.

In [ ]:
# The test loader is first used here, after the intervention and both checkpoints are fixed.
write_json(RUN / "final_evaluation_plan.json", {
    "checkpoint_sha256": {name: sha256_file(RUN / "checkpoints" / f"{name}_best.pt")
                          for name in ("baseline", "intervention")},
    "baseline_epoch": BASELINE_EPOCH, "intervention_epoch": INTERVENTION_EPOCH,
    "intervention_mode": MODE, "hypothesis": HYPOTHESIS,
    "threshold": CONFIG["probability_threshold"], "split_sha256": SPLIT_SHA,
})
test_frames = {}
for model_name, model in (("baseline", BASELINE), ("intervention", INTERVENTION)):
    frame = assign_geometry(evaluate(model, "test", detailed=True))
    frame = frame.merge(SPLIT[["id", "group_id"]], on="id", validate="one_to_one")
    test_frames[model_name] = frame
    save_csv(frame, f"test_{model_name}_per_image.csv")

TEST_ROWS = ["dice", "iou", "recall", "precision", "boundary_f1"]
baseline_test, intervention_test = test_frames["baseline"], test_frames["intervention"]
paired = baseline_test.merge(intervention_test[["id", *TEST_ROWS]], on="id", validate="one_to_one",
                             suffixes=("_baseline", "_intervention"))
assert len(paired) == len(SPLIT_IDS["test"])
for metric in TEST_ROWS:
    paired[f"delta_{metric}"] = paired[f"{metric}_intervention"] - paired[f"{metric}_baseline"]
save_csv(paired, "test_paired_per_image.csv")

def paired_comparison(frame, metric, subgroup, seed):
    column = f"delta_{metric}"
    cluster = frame.groupby("group_id")[column].agg(["sum", "count"])
    n_groups = len(cluster)
    lower, upper = np.nan, np.nan
    if n_groups >= 2:
        rng = np.random.default_rng(seed)
        choices = rng.integers(n_groups, size=(CONFIG["bootstrap_repeats"], n_groups))
        bootstrap = cluster["sum"].to_numpy()[choices].sum(axis=1) / cluster["count"].to_numpy()[choices].sum(axis=1)
        lower, upper = np.quantile(bootstrap, [0.025, 0.975])
    return {
        "subgroup": subgroup, "metric": metric, "n": len(frame), "groups": n_groups,
        "baseline": frame[f"{metric}_baseline"].mean(), "intervention": frame[f"{metric}_intervention"].mean(),
        "delta": frame[column].mean(), "ci95_low": lower, "ci95_high": upper,
    }

comparison_rows = [paired_comparison(paired, metric, "all", CONFIG["seed"] + 1200) for metric in TEST_ROWS]
if MODE == "small_positive":
    target_subgroup, target_metric = "small", "recall"
    subset = paired[paired.area_fraction_group.eq(target_subgroup)]
else:
    target_subgroup, target_metric = "complex", "boundary_f1"
    subset = paired[paired.boundary_complexity_group.eq(target_subgroup)]
comparison_rows.append(paired_comparison(subset, target_metric, target_subgroup, CONFIG["seed"] + 1201))
COMPARISON = pd.DataFrame(comparison_rows)
save_csv(COMPARISON, "test_model_comparison.csv")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), gridspec_kw={"width_ratios": [1.15, 1]}, layout="constrained")
all_cases = COMPARISON[COMPARISON.subgroup.eq("all")]
positions = np.arange(len(all_cases))
axes[0].bar(positions - 0.18, all_cases.baseline, width=0.36, color="#2D4865", label="Baseline")
axes[0].bar(positions + 0.18, all_cases.intervention, width=0.36, color="#087E8B", label="Intervention")
axes[0].set(ylim=(0, 1), xticks=positions, xticklabels=["Dice", "IoU", "Recall", "Precision", "Boundary F1"],
            title="Final held-out test", ylabel="Macro score")
axes[0].tick_params(axis="x", labelrotation=20)
axes[0].legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.20), ncol=2)
ax = axes[1]
from matplotlib.ticker import MaxNLocator
ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
metric_labels = dict(zip(TEST_ROWS, ["Dice", "IoU", "Recall", "Precision", "Boundary F1"], strict=True))
for index, row in enumerate(COMPARISON.itertuples()):
    if np.isfinite(row.ci95_low) and np.isfinite(row.ci95_high):
        ax.hlines(index, row.ci95_low, row.ci95_high, color="#087E8B", lw=2)
    if np.isfinite(row.delta):
        ax.plot(row.delta, index, "o", color="#087E8B")
ax.axvline(0, ls="--", lw=1, color="#535B65")
ax.set(yticks=range(len(COMPARISON)), yticklabels=[f"{metric_labels[r.metric]} · {r.subgroup}" for r in COMPARISON.itertuples()],
       xlabel="Intervention minus baseline", title="Paired group bootstrap 95% intervals")
ax.invert_yaxis()
save_figure(fig, "test_comparison.png")
display(COMPARISON.round(4))


## 12 · 보고서와 ZIP 생성

CSV, PNG, 실행 설정, 초기·최적 모델 체크포인트, `RESULTS.md`를 한 ZIP으로 저장합니다. 재개용 마지막 epoch 체크포인트까지 묶으려면 설정 셀의 `INCLUDE_RESUME_CHECKPOINTS_IN_ZIP=True`를 사용합니다. Colab에서 자동 다운로드가 시작되지 않으면 왼쪽 **파일** 탭에서 ZIP을 내려받으세요.

In [ ]:
def format_decimal(value, digits=3):
    return f"{value:.{digits}f}" if pd.notna(value) else "NA"

def markdown_table(frame):
    header = [str(column) for column in frame.columns]
    lines = ["| " + " | ".join(header) + " |", "| " + " | ".join(["---"] * len(header)) + " |"]
    for values in frame.itertuples(index=False, name=None):
        cells = [format_decimal(value) if isinstance(value, (float, np.floating)) else str(value) for value in values]
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)

natural_top = NATURAL_BUDGET.loc[NATURAL_BUDGET.planned_budget.eq(0.2)].iloc[0]
synthetic_top = SYNTHETIC_MEANS.loc[
    SYNTHETIC_MEANS.planned_budget.eq(0.2) & SYNTHETIC_MEANS.scoring.eq("annotation_review_score")
].iloc[0]
natural = NATURAL_STATS.iloc[0]
synthetic_pr = SYNTHETIC_STATS.groupby("scoring", as_index=False)[["prevalence", "average_precision", "auroc"]].mean()
type_summary = SYNTHETIC_TYPES.groupby("corruption", as_index=False).agg(
    n=("n", "sum"), found=("top20_found", "sum"), random_expected=("random_expected_found", "sum"),
    mean_trial_iou=("mean_candidate_iou", "mean"),
)
primary, target = COMPARISON.iloc[0], COMPARISON.iloc[-1]
if pd.isna(primary.ci95_low):
    overall = "그룹 수가 부족해 Dice 차이의 구간을 계산하지 못했습니다."
elif primary.ci95_low > 0:
    overall = "이 분할에서 전체 Dice 차이의 95% 구간은 0보다 높았습니다."
elif primary.ci95_high < 0:
    overall = "이 분할에서 전체 Dice 차이의 95% 구간은 0보다 낮았습니다."
else:
    overall = "이 분할에서 전체 Dice 차이의 95% 구간에 0이 포함됐습니다."
gap_table = pd.DataFrame(EVIDENCE).T.reset_index(names="candidate")
if SMOKE:
    run_status = "**합성 데이터 스모크 테스트입니다. 아래 숫자는 코드 검증용입니다.**"
elif DEVICE.type == "cpu":
    run_status = "**Kvasir-SEG 축소 CPU 검증입니다. Colab 기본 GPU 결과는 별도 실행에서 생성됩니다.**"
else:
    run_status = "**Kvasir-SEG GPU 연구 실행 결과입니다.**"
evidence_status = (
    "선택한 하위군에서 기준 모델의 Dice가 비교군보다 낮았습니다." if DECISION["positive_gap_observed"]
    else "선택 규칙에 따른 탐색 실험입니다. 선택된 하위군에서 양의 Dice 격차를 확인하지 못했습니다."
)
report = f"""# 내시경 용종 분할 실패 분석과 검수 우선순위

{run_status}

실행 ID `{RUN_ID}`, 시드 {CONFIG['seed']}, 장치 `{ENVIRONMENT['gpu'] or DEVICE}`.
입력 {len(PAIRS)}쌍, {CONFIG['side']}×{CONFIG['side']}, 모델별 {CONFIG['epochs']} epoch.
모델 LR-ASPP MobileNetV3-Large {N_PARAMETERS:,}개 파라미터.
상세 설정, 데이터 출처와 체크섬, 환경은 `run_setup.json`에 기록했습니다.

## 분할

{markdown_table(split_summary.reset_index())}

검증 선택셋에서 체크포인트를 골랐고, 검증 분석셋에서 가설과 검수 성능을 탐색했습니다.
테스트셋은 기준 모델과 개선 모델의 최종 성능 측정에 사용했습니다.

## 검증 분석셋의 실패 양상

{markdown_table(DIAGNOSTICS[['feature', 'group', 'n', 'mean_dice', 'failure_rate', 'mean_boundary_f1']])}

실패 기준은 단일 추론 Dice < {CONFIG['failure_dice']}입니다. 크기와 경계 복잡도 구간은 학습셋 삼분위수입니다.
경계 복잡도는 전처리 해상도의 픽셀 격자에서 계산한 형태 지표입니다.

![크기·경계별 실패](figures/validation_failure_by_geometry.png)

![실패 사례](figures/validation_failure_examples.png)

## 선정된 개선 실험

{HYPOTHESIS} {evidence_status}

{markdown_table(gap_table[['candidate', 'group', 'group_n', 'other_n', 'group_dice', 'other_dice', 'dice_gap']])}

초기 체크포인트와 배치 순서, 증강, 최적화 설정, 학습 기간을 공유하고 BCE의 픽셀 가중치만 변경했습니다.
선택된 epoch는 기준 모델 {BASELINE_EPOCH}, 개선 모델 {INTERVENTION_EPOCH}입니다.

![학습 곡선](figures/training_curves.png)

## 예측 실패 검수

{len(NATURAL)}쌍 중 기준 모델 실패 {int(natural.failures)}건.
검수 예산 20%에서 실제 검수 {int(natural_top.reviewed)}건, 불확실성 순위 발견 {int(natural_top.ranked_found)}건,
같은 예산의 무작위 평균 발견 {format_decimal(natural_top.random_mean_found, 2)}건,
무작위 평균 대비 배수 {format_decimal(natural_top.lift_vs_random_mean, 2)}입니다.
Average Precision {format_decimal(natural.average_precision)}, AUROC {format_decimal(natural.auroc)}입니다.
정답 마스크는 저장한 순위의 평가에 사용했습니다.

![예측 실패 검수](figures/validation_prediction_review.png)

## 합성 주석 오류 검수

{CONFIG['injection_repeats']}회 주입 실험에서 검수 예산 20%의 평균 발견 건수는 복합 점수
{format_decimal(synthetic_top.ranked_found, 2)}건, 무작위 {format_decimal(synthetic_top.random_mean_found, 2)}건,
무작위 평균 대비 배수 {format_decimal(synthetic_top.lift_vs_random_mean, 2)}입니다.
시험마다 검수 후보에 이동·침식·팽창·삭제를 주입했습니다. 실제 픽셀이 달라진 경우를 양성으로 집계했습니다.
동일 이미지가 반복에 재사용되므로 아래 수치는 기술 통계로 제시합니다.

{markdown_table(synthetic_pr)}

{markdown_table(type_summary)}

![합성 주석 오류 검수](figures/validation_synthetic_annotation_review.png)

## 최종 테스트

{markdown_table(COMPARISON)}

주 평가 변수는 전체 macro Dice 차이입니다. {overall}
가설 대상인 {target.subgroup}의 {target.metric} 차이는 {format_decimal(target.delta)}입니다.
구간은 이미지 중복 그룹 단위의 paired percentile bootstrap으로 계산했습니다.
학습 초기화와 분할을 반복했을 때의 변동은 이 구간에 포함되지 않습니다.

![최종 비교](figures/test_comparison.png)

## 해석 범위

공개 배포본에서 환자나 검사 세션 식별자를 제공하지 않아 이미지 유사도로 중복 후보만 묶었습니다.
탐지하지 못한 근접 프레임이나 환자 내 상관이 남을 수 있습니다. 실패 유형 간 관계는 탐색적 연관입니다.
검수 큐는 원본 정답 없이 계산할 수 있습니다. 모델이 일관되게 틀린 사례는 엔트로피가 낮을 수 있고,
합성 오류 분포와 실제 주석 오류 분포의 차이도 남습니다. 임상 성능과 환자 안전성은 평가 범위 밖입니다.

## 데이터셋 인용

Jha et al. (2020). *Kvasir-SEG: A Segmented Polyp Dataset*. MultiMedia Modeling, pp. 451–462.
[논문]({PAPER}) · [공식 데이터셋 및 이용 조건]({OFFICIAL_PAGE})
"""
(RUN / "RESULTS.md").write_text(report, encoding="utf-8")

archive = ROOT / f"kvasir_seg_results_{PROFILE}_{RUN_ID}.zip"
temporary = archive.with_suffix(".zip.part")
with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=3) as package:
    for path in sorted(RUN.rglob("*")):
        if not path.is_file() or path.name.endswith(".part"):
            continue
        if path.name.endswith("_last.pt") and not INCLUDE_RESUME_CHECKPOINTS_IN_ZIP:
            continue
        package.write(path, path.relative_to(RUN.parent))
with zipfile.ZipFile(temporary) as package:
    if package.testzip() is not None:
        raise RuntimeError("결과 ZIP 무결성 검사 실패. 이 셀을 다시 실행하세요.")
temporary.replace(archive)

from IPython.display import Markdown, display
display(Markdown(report.split("## 분할", 1)[0]))
print(f"Epoch resume checkpoints: {RUN / 'checkpoints'}")
print(f"Report: {RUN / 'RESULTS.md'}")
print(f"Results ZIP: {archive} ({archive.stat().st_size / 2**20:.1f} MiB)")
if IN_COLAB and DOWNLOAD_RESULTS:
    from google.colab import files
    try:
        files.download(str(archive))
    except Exception as error:
        print(f"브라우저 자동 다운로드를 시작할 수 없습니다 ({error}). Colab 파일 탭에서 ZIP을 내려받으세요.")
